[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/094.%20The%20Multi-Echelon%20Inventory%20Optimization%20Problem/P94-Tier-3_executed.ipynb)

---


# 94. The Multi-Echelon Inventory Optimization Problem

## Tier 3: Moth-Flame Optimization

### Purpose and scope

This notebook uses Moth-Flame Optimization (MFO), a population-based metaheuristic, to search for a low-cost base-stock policy across a small, multi-echelon network. A **base-stock level** is the target inventory quantity assigned to a location. Each candidate policy is a vector with one continuous decision variable per location. Its fitness is the expected holding and backorder cost over the supplied demand scenarios.

The notebook demonstrates the MFO search process, evaluates its convergence, compares its result with a random-search baseline, and explores how population size and iteration count affect one run. MFO is a heuristic: it does not certify that the best policy it finds is globally optimal. Because the algorithm uses random numbers and no seed is specified, results may vary between executions.

### Model assumptions

- The decision variables are continuous base-stock levels constrained by location-specific bounds; inventory quantities are not rounded to integers.
- Demand is represented by a finite set of scenarios with probabilities that sum to one. The objective is expected cost under those scenarios.
- A retail location faces its own scenario demand. A location at a higher echelon faces the sum of scenario demand at all locations with a lower echelon level. The model evaluates each echelon's inventory cost against that assigned demand; it does not simulate material flows between nodes.
- Holding and backorder costs are linear in end-of-period surplus and shortage, respectively. The model is a one-period scenario calculation, not a time-phased inventory simulation.
- Lead times are included in the problem data but are not used in the objective or search. Replenishment timing, capacity, service-level constraints, ordering costs, and demand evolution are outside this model's scope.

### MFO search, step by step

1. **Initialize moths.** Generate candidate base-stock vectors uniformly within the specified bounds.
2. **Evaluate fitness.** Compute each candidate's expected cost over the demand scenarios; lower cost is better.
3. **Select flames.** Rank candidates by cost and use the best candidates as flame targets. The number of flames decreases during the run.
4. **Move moths.** Move each candidate relative to its assigned flame using a randomized logarithmic-spiral step, then enforce the decision-variable bounds.
5. **Retain improvements.** A moth adopts its new position only when that position has lower cost.
6. **Record and report.** Track the best cost and flame count across iterations, then return the best policy found.

A flame is a promising candidate solution that attracts moths during the search. The spiral movement provides a way to explore positions around a flame; reducing the number of flames concentrates the search on fewer targets as iterations proceed. These mechanisms do not guarantee global optimality.

### Mathematical model

Let $I$ be the set of locations, $S$ the set of demand scenarios, and $x_i$ the base-stock decision for location $i$. For scenario $s$, let $q_{is}$ be the demand assigned to location $i$ under the echelon rule above, and let $p_s$ be the scenario probability. Define the surplus and shortage as $h_{is}=\max(x_i-q_{is},0)$ and $b_{is}=\max(q_{is}-x_i,0)$, respectively. The fitness function is the expected total cost:

$$\min_{x}\quad \sum_{s\in S} p_s \sum_{i\in I}\left(c_i^{H}h_{is}+c_i^{B}b_{is}\right)$$

subject to the location-specific search bounds

$$\ell_i\leq x_i\leq u_i\qquad \text{for all }i\in I.$$

Here, $c_i^{H}$ and $c_i^{B}$ are the holding and backorder cost rates, and $\ell_i$ and $u_i$ are the lower and upper bounds. The code evaluates this objective in `evaluate_cost_breakdown` and `evaluate_fitness`; the bounds are enforced in `logarithmic_spiral`. Since the search variables are continuous, this formulation does not impose integer inventory quantities.

### MFO update used in this notebook

For moth position $X_i$ and assigned flame position $F_j$, the component-wise distance is $D_i=|F_j-X_i|$. A new position is generated as

$$X_i^{\mathrm{new}}=F_j+D_i e^{bt}\cos(2\pi t),\qquad t\sim\mathcal{U}[-1,1],\quad b=1.$$

The result is clipped to the location bounds. Here, $t$ is a random spiral parameter and $b$ controls the spiral shape. The implementation retains a moth's current position unless the new position has lower fitness. Flame assignments follow the notebook's index-based wrap-around rule. These choices define the particular MFO implementation demonstrated here.

### Interpreting the analyses

The convergence curve shows the best cost recorded at each iteration; a non-increasing curve is expected because moths retain improvements. The flame plot shows how many candidate targets are used. The random-search comparison is a stochastic reference, not an optimality benchmark: a single run of either method does not establish which method is generally better. Sensitivity results also come from individual stochastic runs and should be interpreted as illustrative, not as a controlled statistical comparison.

The solution summary reports the base-stock policy and expected cost components under the model. It does not estimate safety stock: that quantity is not separately defined by this one-period formulation. The final fitness is a best-found heuristic objective value, not a proven optimum.

### Computational considerations

With $N$ moths, $T$ iterations, $|I|$ locations, and $|S|$ scenarios, fitness evaluation takes $O(|I||S|)$ work, so the main search takes approximately $O(NT|I||S|)$ work. The random-search baseline requires $O(R|I||S|)$ work for $R$ evaluations. MFO has no optimality guarantee; its observed solution quality depends on the bounds, parameters, and random draws.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

In [ ]:
@dataclass
class MEIOProblem:
    """Data and dimensions for a multi-echelon inventory problem."""

    location_names: List[str]
    echelon_levels: List[int]
    holding_costs: List[float]
    backorder_costs: List[float]
    lead_times: List[int]
    demand_scenarios: List[Dict[str, float]]
    scenario_probabilities: List[float]
    variable_bounds: List[Tuple[float, float]]

    def __post_init__(self):
        self.num_locations = len(self.location_names)
        self.num_scenarios = len(self.demand_scenarios)

        location_data_lengths = [
            len(self.echelon_levels),
            len(self.holding_costs),
            len(self.backorder_costs),
            len(self.lead_times),
            len(self.variable_bounds),
        ]
        if any(length != self.num_locations for length in location_data_lengths):
            raise ValueError("Each location must have one value for every location parameter.")
        if self.num_scenarios == 0:
            raise ValueError("At least one demand scenario is required.")
        if len(self.scenario_probabilities) != self.num_scenarios:
            raise ValueError("Each demand scenario must have one probability.")
        if not np.isclose(sum(self.scenario_probabilities), 1.0):
            raise ValueError("Scenario probabilities must sum to one.")
        if any(probability < 0 for probability in self.scenario_probabilities):
            raise ValueError("Scenario probabilities cannot be negative.")
        if any(lower > upper for lower, upper in self.variable_bounds):
            raise ValueError("Each lower bound must be no greater than its upper bound.")


@dataclass
class Moth:
    """A candidate inventory policy and its objective value."""

    position: np.ndarray
    fitness: float = float("inf")


@dataclass
class MFOResults:
    """Summary returned by a Moth-Flame Optimization run."""

    best_solution: np.ndarray
    best_fitness: float
    convergence_history: List[float]
    flame_history: List[int]
    computation_time: float
    final_moths: List[Moth]

In [ ]:
class MothFlameOptimizer:
    """Moth-Flame Optimization for the specified inventory-cost model."""

    def __init__(
        self,
        problem: MEIOProblem,
        population_size: int = 30,
        max_iterations: int = 100,
    ):
        if population_size < 1:
            raise ValueError("Population size must be at least one.")
        if max_iterations < 1:
            raise ValueError("Maximum iterations must be at least one.")

        self.problem = problem
        self.population_size = population_size
        self.max_iterations = max_iterations
        self.moths = []
        self.flames = []

    def assigned_demand(self, scenario: Dict[str, float], loc_idx: int) -> float:
        """Return demand assigned to a location by the echelon rule."""
        location_name = self.problem.location_names[loc_idx]
        echelon = self.problem.echelon_levels[loc_idx]

        if echelon == 0:
            return scenario.get(location_name, 0.0)

        return sum(
            scenario.get(name, 0.0)
            for name, downstream_echelon in zip(
                self.problem.location_names,
                self.problem.echelon_levels,
            )
            if downstream_echelon < echelon
        )

    def evaluate_cost_breakdown(self, position: np.ndarray) -> Dict[str, np.ndarray]:
        """Compute expected holding and backorder cost by location."""
        if len(position) != self.problem.num_locations:
            raise ValueError("A policy must contain one stock level per location.")

        expected_holding_costs = np.zeros(self.problem.num_locations)
        expected_backorder_costs = np.zeros(self.problem.num_locations)

        for scenario, probability in zip(
            self.problem.demand_scenarios,
            self.problem.scenario_probabilities,
        ):
            for loc_idx in range(self.problem.num_locations):
                demand = self.assigned_demand(scenario, loc_idx)
                stock = position[loc_idx]
                surplus = max(stock - demand, 0.0)
                shortage = max(demand - stock, 0.0)

                expected_holding_costs[loc_idx] += (
                    probability
                    * surplus
                    * self.problem.holding_costs[loc_idx]
                )
                expected_backorder_costs[loc_idx] += (
                    probability
                    * shortage
                    * self.problem.backorder_costs[loc_idx]
                )

        return {
            "holding_costs": expected_holding_costs,
            "backorder_costs": expected_backorder_costs,
        }

    def evaluate_fitness(self, position: np.ndarray) -> float:
        """Return expected total cost; lower fitness is preferred."""
        cost_breakdown = self.evaluate_cost_breakdown(position)
        return float(
            np.sum(cost_breakdown["holding_costs"])
            + np.sum(cost_breakdown["backorder_costs"])
        )

    def initialize_population(self):
        """Sample and evaluate the initial moth population."""
        self.moths = []

        for _ in range(self.population_size):
            position = np.array(
                [
                    random.uniform(lower, upper)
                    for lower, upper in self.problem.variable_bounds
                ],
                dtype=float,
            )
            moth = Moth(position=position)
            moth.fitness = self.evaluate_fitness(position)
            self.moths.append(moth)

    def update_flames(self, iteration: int) -> int:
        """Rank moths and select the iteration's flame targets."""
        sorted_moths = sorted(self.moths, key=lambda moth: moth.fitness)
        flame_num = int(
            np.round(
                self.population_size
                - iteration
                * (self.population_size - 1)
                / self.max_iterations
            )
        )
        flame_num = max(1, flame_num)
        self.flames = sorted_moths[:flame_num]
        return flame_num

    def logarithmic_spiral(
        self,
        moth_pos: np.ndarray,
        flame_pos: np.ndarray,
        iteration: int,
        max_iterations: int,
    ) -> np.ndarray:
        """Generate a bounded candidate position around a flame."""
        distance = np.abs(flame_pos - moth_pos)
        t = random.uniform(-1.0, 1.0)
        b = 1.0
        spiral_factor = np.exp(b * t) * np.cos(2.0 * np.pi * t)
        new_position = flame_pos + distance * spiral_factor

        for idx, (lower, upper) in enumerate(self.problem.variable_bounds):
            new_position[idx] = np.clip(new_position[idx], lower, upper)

        return new_position

    def optimize(self) -> MFOResults:
        """Run MFO and return the best policy found during the run."""
        start_time = time.time()
        self.initialize_population()

        convergence_history = []
        flame_history = []

        for iteration in range(self.max_iterations):
            flame_num = self.update_flames(iteration)
            flame_history.append(flame_num)

            # Snapshot targets so updates to moths cannot move a flame mid-iteration.
            flame_positions = [flame.position.copy() for flame in self.flames]

            for moth_idx, moth in enumerate(self.moths):
                flame_idx = moth_idx % flame_num
                new_position = self.logarithmic_spiral(
                    moth.position,
                    flame_positions[flame_idx],
                    iteration,
                    self.max_iterations,
                )
                new_fitness = self.evaluate_fitness(new_position)

                if new_fitness < moth.fitness:
                    moth.position = new_position
                    moth.fitness = new_fitness

            convergence_history.append(min(moth.fitness for moth in self.moths))

        best_moth = min(self.moths, key=lambda moth: moth.fitness)
        end_time = time.time()

        return MFOResults(
            best_solution=best_moth.position.copy(),
            best_fitness=best_moth.fitness,
            convergence_history=convergence_history,
            flame_history=flame_history,
            computation_time=end_time - start_time,
            final_moths=[
                Moth(position=moth.position.copy(), fitness=moth.fitness)
                for moth in self.moths
            ],
        )

In [ ]:
# Define the network, costs, scenarios, and search bounds.
location_names = ["Retail_A", "Retail_B", "DC1", "Central_Warehouse"]
echelon_levels = [0, 0, 1, 2]
holding_costs = [2.0, 2.0, 1.5, 1.0]
backorder_costs = [10.0, 10.0, 5.0, 2.0]
lead_times = [2, 2, 5, 3]

demand_scenarios = [
    {"Retail_A": 10, "Retail_B": 15},
    {"Retail_A": 20, "Retail_B": 30},
    {"Retail_A": 30, "Retail_B": 45},
]
scenario_probabilities = [0.25, 0.50, 0.25]

variable_bounds = [
    (0, 80),
    (0, 100),
    (0, 150),
    (0, 200),
]

problem = MEIOProblem(
    location_names=location_names,
    echelon_levels=echelon_levels,
    holding_costs=holding_costs,
    backorder_costs=backorder_costs,
    lead_times=lead_times,
    demand_scenarios=demand_scenarios,
    scenario_probabilities=scenario_probabilities,
    variable_bounds=variable_bounds,
)

population_size = 30
max_iterations = 100

print("Multi-Echelon Inventory Optimization Problem")
print("=" * 48)
print(f"Locations: {problem.location_names}")
print(f"Echelon levels: {problem.echelon_levels}")
print(f"Demand scenarios: {problem.num_scenarios}")
print(f"Population size: {population_size} moths")
print(f"Maximum iterations: {max_iterations}")
print("\nDecision-variable bounds:")
for name, bounds in zip(problem.location_names, problem.variable_bounds):
    print(f"  {name}: [{bounds[0]}, {bounds[1]}] units")

In [ ]:
# Run the MFO search and report its best-found policy.
optimizer = MothFlameOptimizer(
    problem,
    population_size=population_size,
    max_iterations=max_iterations,
)

print("Running Moth-Flame Optimization...")
results = optimizer.optimize()

print("\nMOTH-FLAME OPTIMIZATION RESULTS")
print(f"Computation time: {results.computation_time:.2f} seconds")
print(f"Best-found expected cost: ${results.best_fitness:.2f}")
print("\nBest-found base-stock policy:")
for name, stock_level in zip(problem.location_names, results.best_solution):
    print(f"  {name}: {stock_level:.2f} units")

initial_recorded_cost = results.convergence_history[0]
final_recorded_cost = results.convergence_history[-1]
if not np.isclose(initial_recorded_cost, 0.0):
    improvement = (
        (initial_recorded_cost - final_recorded_cost)
        / initial_recorded_cost
        * 100.0
    )
    print(
        f"\nRecorded best cost changed from ${initial_recorded_cost:.2f} "
        f"to ${final_recorded_cost:.2f} ({improvement:.1f}% improvement)."
    )
else:
    print("\nThe first recorded best cost is zero; a percentage change is undefined.")

In [ ]:
def analyze_convergence(convergence_history, flame_history):
    """Summarize recorded cost changes and flame counts."""
    if not convergence_history:
        raise ValueError("Convergence history cannot be empty.")

    improvements = []
    for previous_cost, current_cost in zip(
        convergence_history[:-1],
        convergence_history[1:],
    ):
        if np.isclose(previous_cost, 0.0):
            improvement = 0.0
        else:
            improvement = (previous_cost - current_cost) / previous_cost
        improvements.append(improvement)

    significant_improvements = sum(
        improvement > 0.01 for improvement in improvements
    )
    total_improvement = (
        0.0
        if np.isclose(convergence_history[0], 0.0)
        else (convergence_history[0] - convergence_history[-1])
        / convergence_history[0]
    )
    window = min(10, len(convergence_history))
    stable = significant_improvements < len(improvements) * 0.1

    return {
        "total_improvement": total_improvement,
        "significant_improvements": significant_improvements,
        "final_window_average": float(np.mean(convergence_history[-window:])),
        "stability_indicator": stable,
        "initial_flames": flame_history[0] if flame_history else 0,
        "final_flames": flame_history[-1] if flame_history else 0,
    }


convergence_analysis = analyze_convergence(
    results.convergence_history,
    results.flame_history,
)

print("CONVERGENCE ANALYSIS")
print(
    f"Recorded best-cost improvement: "
    f"{convergence_analysis['total_improvement'] * 100:.1f}%"
)
print(
    "Iterations with an improvement greater than 1%: "
    f"{convergence_analysis['significant_improvements']}"
)
print(
    "Average cost over the final recorded window: "
    f"${convergence_analysis['final_window_average']:.2f}"
)
print(
    "Stability indicator (few significant improvements): "
    f"{'Yes' if convergence_analysis['stability_indicator'] else 'No'}"
)
print("\nFlame counts:")
print(f"  First recorded count: {convergence_analysis['initial_flames']}")
print(f"  Final recorded count: {convergence_analysis['final_flames']}")

In [ ]:
def random_search_baseline(
    problem: MEIOProblem,
    num_evaluations: int = 3000,
) -> Dict:
    """Sample bounded policies and return the lowest-cost sample."""
    if num_evaluations < 1:
        raise ValueError("The number of evaluations must be at least one.")

    evaluator = MothFlameOptimizer(problem, population_size=1, max_iterations=1)
    best_solution = None
    best_fitness = float("inf")
    fitness_history = []

    for _ in range(num_evaluations):
        position = np.array(
            [
                random.uniform(lower, upper)
                for lower, upper in problem.variable_bounds
            ],
            dtype=float,
        )
        fitness = evaluator.evaluate_fitness(position)
        fitness_history.append(fitness)

        if fitness < best_fitness:
            best_fitness = fitness
            best_solution = position.copy()

    return {
        "best_solution": best_solution,
        "best_fitness": best_fitness,
        "fitness_history": fitness_history,
    }


print("Running random-search baseline...")
random_results = random_search_baseline(problem, num_evaluations=3000)

print("\nSINGLE-RUN COMPARISON WITH RANDOM SEARCH")
print(f"MFO best-found cost: ${results.best_fitness:.2f}")
print(f"Random-search best-found cost: ${random_results['best_fitness']:.2f}")
if not np.isclose(random_results["best_fitness"], 0.0):
    relative_difference = (
        random_results["best_fitness"] - results.best_fitness
    ) / random_results["best_fitness"] * 100.0
    print(
        "MFO cost reduction relative to the random-search result: "
        f"{relative_difference:.1f}%"
    )
else:
    print("Relative cost difference is undefined because the baseline cost is zero.")

print("\nPolicy comparison (units):")
print(f"{'Location':<22} {'MFO':>10} {'Random search':>16} {'Difference':>12}")
print("-" * 62)
for idx, name in enumerate(problem.location_names):
    mfo_value = results.best_solution[idx]
    random_value = random_results["best_solution"][idx]
    print(
        f"{name:<22} {mfo_value:>10.2f} "
        f"{random_value:>16.2f} {mfo_value - random_value:>12.2f}"
    )

In [ ]:
# Visualize convergence, flame counts, policies, and final-population costs.
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Moth-Flame Optimization: Multi-Echelon Inventory Analysis",
    fontsize=16,
    fontweight="bold",
)

iterations = range(1, len(results.convergence_history) + 1)

ax1 = axes[0, 0]
ax1.plot(iterations, results.convergence_history, color="tab:blue", linewidth=2)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Best Expected Cost ($)")
ax1.set_title("Best-So-Far Convergence")
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(
    iterations,
    results.flame_history,
    color="tab:red",
    linewidth=2,
    marker="o",
    markersize=3,
)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Number of Flames")
ax2.set_title("Adaptive Flame Count")
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
x_positions = np.arange(problem.num_locations)
bar_width = 0.35
ax3.bar(
    x_positions - bar_width / 2,
    results.best_solution,
    bar_width,
    label="MFO",
    color="#2E86AB",
)
ax3.bar(
    x_positions + bar_width / 2,
    random_results["best_solution"],
    bar_width,
    label="Random search",
    color="#A23B72",
)
ax3.set_xlabel("Location")
ax3.set_ylabel("Base-Stock Level (units)")
ax3.set_title("Best-Found Policy Comparison")
ax3.set_xticks(x_positions)
ax3.set_xticklabels(problem.location_names, rotation=35, ha="right")
ax3.legend()
ax3.grid(True, axis="y", alpha=0.3)

ax4 = axes[1, 1]
final_fitness = [moth.fitness for moth in results.final_moths]
ax4.hist(final_fitness, bins=15, alpha=0.7, color="#F18F01", edgecolor="black")
ax4.axvline(
    results.best_fitness,
    color="tab:red",
    linestyle="--",
    linewidth=2,
    label=f"MFO best: ${results.best_fitness:.2f}",
)
ax4.axvline(
    random_results["best_fitness"],
    color="tab:green",
    linestyle="--",
    linewidth=2,
    label=f"Random-search best: ${random_results['best_fitness']:.2f}",
)
ax4.set_xlabel("Expected Cost ($)")
ax4.set_ylabel("Number of Moths")
ax4.set_title("Final MFO Population Fitness")
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def parameter_sensitivity_analysis(base_problem: MEIOProblem):
    """Run single MFO trials over selected population and iteration settings."""
    population_sizes = [10, 20, 30, 50]
    population_results = {}

    for population_size in population_sizes:
        trial_optimizer = MothFlameOptimizer(
            base_problem,
            population_size=population_size,
            max_iterations=50,
        )
        trial = trial_optimizer.optimize()
        first_cost = trial.convergence_history[0]
        final_cost = trial.convergence_history[-1]
        improvement = (
            0.0
            if np.isclose(first_cost, 0.0)
            else (first_cost - final_cost) / first_cost
        )
        population_results[population_size] = {
            "best_fitness": trial.best_fitness,
            "computation_time": trial.computation_time,
            "final_improvement": improvement,
        }

    iteration_counts = [25, 50, 100, 200]
    iteration_results = {}

    for iteration_count in iteration_counts:
        trial_optimizer = MothFlameOptimizer(
            base_problem,
            population_size=30,
            max_iterations=iteration_count,
        )
        trial = trial_optimizer.optimize()
        first_cost = trial.convergence_history[0]
        final_cost = trial.convergence_history[-1]
        improvement = (
            0.0
            if np.isclose(first_cost, 0.0)
            else (first_cost - final_cost) / first_cost
        )
        iteration_results[iteration_count] = {
            "best_fitness": trial.best_fitness,
            "computation_time": trial.computation_time,
            "final_improvement": improvement,
        }

    return population_results, iteration_results


print("Running parameter sensitivity analysis...")
pop_sensitivity, iter_sensitivity = parameter_sensitivity_analysis(problem)

print("\nPOPULATION-SIZE TRIALS")
for population_size, result in pop_sensitivity.items():
    print(
        f"Population {population_size:2d}: "
        f"cost=${result['best_fitness']:.2f}, "
        f"time={result['computation_time']:.2f} seconds, "
        f"recorded improvement={result['final_improvement'] * 100:.1f}%"
    )

print("\nITERATION-COUNT TRIALS")
for iteration_count, result in iter_sensitivity.items():
    print(
        f"Iterations {iteration_count:3d}: "
        f"cost=${result['best_fitness']:.2f}, "
        f"time={result['computation_time']:.2f} seconds, "
        f"recorded improvement={result['final_improvement'] * 100:.1f}%"
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("MFO Parameter Sensitivity: Single-Run Results", fontsize=16)

population_sizes = list(pop_sensitivity)
population_costs = [
    pop_sensitivity[size]["best_fitness"] for size in population_sizes
]
population_times = [
    pop_sensitivity[size]["computation_time"] for size in population_sizes
]

ax1 = axes[0, 0]
ax1.plot(population_sizes, population_costs, "o-", color="#2E86AB")
ax1.set_title("Best-Found Cost vs Population Size")
ax1.set_xlabel("Population Size (moths)")
ax1.set_ylabel("Best-Found Expected Cost ($)")
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(population_sizes, population_times, "s-", color="#A23B72")
ax2.set_title("Computation Time vs Population Size")
ax2.set_xlabel("Population Size (moths)")
ax2.set_ylabel("Computation Time (seconds)")
ax2.grid(True, alpha=0.3)

iteration_counts = list(iter_sensitivity)
iteration_costs = [
    iter_sensitivity[count]["best_fitness"] for count in iteration_counts
]
iteration_times = [
    iter_sensitivity[count]["computation_time"] for count in iteration_counts
]

ax3 = axes[1, 0]
ax3.plot(iteration_counts, iteration_costs, "o-", color="#F18F01")
ax3.set_title("Best-Found Cost vs Iteration Count")
ax3.set_xlabel("Maximum Iterations")
ax3.set_ylabel("Best-Found Expected Cost ($)")
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 1]
ax4.plot(iteration_counts, iteration_times, "s-", color="#C73E1D")
ax4.set_title("Computation Time vs Iteration Count")
ax4.set_xlabel("Maximum Iterations")
ax4.set_ylabel("Computation Time (seconds)")
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Check that the reported objective matches the policy and summarize its cost.
cost_breakdown = optimizer.evaluate_cost_breakdown(results.best_solution)
holding_cost_total = float(np.sum(cost_breakdown["holding_costs"]))
backorder_cost_total = float(np.sum(cost_breakdown["backorder_costs"]))
recomputed_fitness = holding_cost_total + backorder_cost_total

assert np.isclose(recomputed_fitness, results.best_fitness)
assert len(results.best_solution) == problem.num_locations
for stock_level, (lower, upper) in zip(
    results.best_solution,
    problem.variable_bounds,
):
    assert lower <= stock_level <= upper

print("FINAL POLICY AND EXPECTED COST SUMMARY")
print("\nBest-found base-stock policy:")
for idx, (name, stock_level) in enumerate(
    zip(problem.location_names, results.best_solution)
):
    print(
        f"{name} (echelon {problem.echelon_levels[idx]}): "
        f"{stock_level:.2f} units"
    )

print(f"\nTotal base-stock quantity: {np.sum(results.best_solution):.2f} units")
print(f"Expected holding cost: ${holding_cost_total:.2f}")
print(f"Expected backorder cost: ${backorder_cost_total:.2f}")
print(f"Expected total cost: ${recomputed_fitness:.2f}")

print("\nInterpretation:")
print(
    "The policy is the lowest-cost candidate retained by this MFO run "
    "under the stated scenario model."
)
print(
    "The stability indicator describes the recorded search history; "
    "it is not a proof of convergence to a global optimum."
)

## Key Insights from the MFO Analysis

### Search behavior

MFO maintains a population of candidate inventory policies and uses ranked flame targets to guide randomized spiral moves. The best-so-far curve and flame-count plot describe how this particular run progressed. Because the algorithm retains improvements and uses a finite number of iterations, a flat section of the curve indicates no recorded improvement during that section; it does not establish global optimality.

### Solution quality and the baseline

The random-search comparison contrasts two stochastic outcomes on the same objective and bounds. A result from one run is not evidence that one search procedure will consistently outperform the other. Repeated trials would be needed for a statistical comparison; this notebook reports individual runs.

### Parameter sensitivity

Population-size and iteration-count trials illustrate how the observed cost and computation time vary with these settings. Each point is a single unseeded run, so variation may reflect both the setting and random initialization. Interpret the plots as exploratory rather than as a definitive ranking of parameter choices.

### Limitations and intended use

- MFO is a heuristic and provides no certificate of optimality.
- Outcomes can vary with random initialization and algorithm settings.
- This objective uses finite scenarios and a one-period surplus-and-shortage calculation.
- Lead times are stored but are not modeled in the cost calculation.
- The network model does not enforce flow balance, capacity, service-level, or replenishment constraints.

The method is useful here as an example of population-based search for a bounded, simulation-evaluated inventory policy. Applying the result to an operational system would require a model that captures the relevant timing, flows, and business constraints.